In [15]:
import torch
import torch.nn as nn
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision import transforms #数据预处理
import matplotlib as plt

In [16]:
train_path = r'C:\Users\35439\Desktop\deepTA_data\ImageNet\train'
valid_path = r'C:\Users\35439\Desktop\deepTA_data\ImageNet\valid'

数据加载器

In [17]:
data_transform = transforms.Compose([
    transforms.RandomResizedCrop(size=(128.128),antialias=True),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])

In [21]:
train_dataset = ImageFolder(train_path,transform=data_transform)

In [22]:
data_loader = DataLoader(train_dataset,batch_size=4,shuffle=True)

In [23]:
for data, label in data_loader:
    print(data.shape,label.shape)

torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size([4, 3, 128, 128]) torch.Size([4])
torch.Size

In [31]:
class ResBlock(nn.Module):
    def __init__(self, in_channels, hid_channels):
        super().__init__()
        self.in_channels = in_channels
        self.hid_channels = hid_channels
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels,hid_channels,kernel_size=3,padding=1),
            nn.BatchNorm2d(hid_channels),
            nn.SiLU(),
            nn.Conv2d(hid_channels,hid_channels,kernel_size=5,padding=2),
            nn.BatchNorm2d(hid_channels),
            nn.SiLU(),
            nn.Conv2d(hid_channels,in_channels,kernel_size=3,padding=1)
        )
    def forward(self,x):
        return x + self.cnn(x)

class ResNet(nn.Module):
    def __init__(self, in_channels, out_channels, res_channels, hid_channels, num_res):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.res_channels = res_channels
        self.hid_channels = hid_channels
        self.num_res = num_res
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels,res_channels,kernel_size=3,padding=1),
            nn.SiLU()
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(res_channels,out_channels,kernel_size=3,padding=1)
        )
        self.hid_res_layers = nn.Sequential(
            *[ResBlock(res_channels,hid_channels) for _ in range(num_res)]
        )
    
    def forward(self,x):
        y = self.encoder(x)
        y = self.hid_res_layers(y)
        y = self.decoder(y)
        return y.mean(dim=[-2,-1])

In [32]:
x = torch.randn([1,3,256,256])
resnet = ResNet(3,20,16,32,20)
y = resnet(x)
print(y.shape)

torch.Size([1, 20])


In [ ]:
def train(model,dataloader,epoch,batch,lr):
    
    optim = torch.optim.Adam(model.patameters(),lr=lr)
    loss_fun = nn.CrossEntropyLoss()
    loss_values = torch.zeros(epoch)

    for i in range(epoch):
        model.train()
        for data, label in dataloader:
            optim.zero_grad()
            pred = model(data)
            loss = loss_fun(pred,label)
            

